# Newton's Method in the Complex Plane

The fundamental theorem of algebra guarantees roots; it does not tell us how to find them. For degree $5$ and higher there is no formula in radicals, so in practice roots are computed **iteratively**. Newton's method, $z_{k+1} = z_k - p(z_k)/p'(z_k)$, carries over from the real line to $\mathbb{C}$ word for word, and it converges to complex roots just as fast as it converges to real ones. But in the plane a new question becomes visible: *which* root does a starting point lead to? Colouring each starting point by its destination produces the **Newton fractal**, whose basin boundaries are infinitely intricate and where an arbitrarily small change of the starting point changes the answer.

This is notebook 03 of module 02 (**Polynomials and roots**) in the **Learning Complex Numbers** series.

## 1. Introduction

Every numerical root finder you use, from `np.roots` (which actually computes eigenvalues of a companion matrix) to the solvers inside circuit simulators and physics engines, rests on the idea behind Newton's method: replace the function by its linear approximation and solve the linear problem instead. On the real line that is the tangent-line picture from calculus. In $\mathbb{C}$ there is no tangent *line* to draw, since the graph of $p: \mathbb{C} \to \mathbb{C}$ is four-dimensional, but the algebra is identical, and the derivative $p'(z_k)$ is a complex number that tells us how $p$ locally **rotates and scales** around $z_k$.

Two things change when we move to the plane:

1. Starting from a real number, Newton's method on a real polynomial can never leave the real line, so it can never find a complex root. Complex starting points are needed.
2. The set of starting points that lead to a given root, its **basin of attraction**, turns out to have a fractal boundary. Cayley asked about this in 1879 for $z^3 - 1$; the answer had to wait for computer pictures a century later.

**Prerequisites**

- Newton's method on the real line, derived from the tangent line: [Newton's method](../../../LearningCalculus/coursera/05_newtons_method/00_newtons_method.ipynb) (LearningCalculus).
- Roots, multiplicity and the fundamental theorem of algebra: [The fundamental theorem of algebra](02_fundamental_theorem_of_algebra.ipynb).
- Complex division and modulus: [Conjugate, modulus and division](../00_foundations/03_conjugate_modulus_and_division.ipynb).
- Cube roots of unity $1, \omega, \bar\omega$ and rotation by $2\pi/3$: [Roots of unity and n-th roots](../01_polar_form_and_euler/05_roots_of_unity_and_nth_roots.ipynb).

After completing this notebook you will be able to:

- Derive and implement Newton's iteration for complex polynomials, scalar and vectorized over a grid.
- Recognize quadratic convergence near a simple root, and the slow linear convergence near a multiple root, and fix the latter.
- Explain why real starting points cannot reach complex roots of real polynomials.
- Render and interpret the Newton fractal of $z^3 - 1$, and demonstrate sensitive dependence on initial conditions near basin boundaries.

## 2. Theory

### 2.1 The iteration

Polynomials are differentiated in $\mathbb{C}$ by the same rules as in $\mathbb{R}$: $(z^n)' = nz^{n-1}$, and $p(z_k + h) = p(z_k) + p'(z_k)h + O(h^2)$ for complex $h$ (Taylor expansion of a polynomial is pure algebra). Module 05 develops complex derivatives in general; for polynomials nothing new is needed.

Newton's idea: near $z_k$, replace $p$ by its linearization and find where *that* vanishes:

$$
p(z_k) + p'(z_k)(z - z_k) = 0 \quad\Longrightarrow\quad z_{k+1} = z_k - \frac{p(z_k)}{p'(z_k)} =: N(z_k).
$$

The map $N(z) = z - p(z)/p'(z)$ is the **Newton map** of $p$. Its fixed points $N(r) = r$ are exactly the roots of $p$ (where $p'(r) \ne 0$). The step $-p(z_k)/p'(z_k)$ is a complex number: its direction is chosen so that, to first order, $p$ moves from $p(z_k)$ straight to $0$.

### 2.2 Quadratic convergence at a simple root

Let $r$ be a simple root ($p(r) = 0$, $p'(r) \ne 0$) and $e_k = z_k - r$. Taylor-expanding around $r$,

$$
p(z_k) = p'(r)e_k + \tfrac12 p''(r)e_k^2 + O(e_k^3), \qquad p'(z_k) = p'(r) + p''(r)e_k + O(e_k^2),
$$

and a short computation gives

$$
e_{k+1} = e_k - \frac{p(z_k)}{p'(z_k)} = \frac{p''(r)}{2p'(r)}\, e_k^2 + O(e_k^3).
$$

The error is **squared** each step: once $|e_k|$ is small, the number of correct digits roughly **doubles** per iteration. Equivalently, $N'(r) = 0$, since in general

$$
N'(z) = \frac{p(z)\, p''(z)}{p'(z)^2}.
$$

A fixed point with $N'(r) = 0$ is called **superattracting**.

### 2.3 Multiple roots: linear convergence

If $r$ has multiplicity $m \ge 2$, write $p(z) = (z - r)^m g(z)$ with $g(r) \ne 0$. Then $\frac{p}{p'} = \frac{(z - r)g}{mg + (z - r)g'}$, and

$$
e_{k+1} \approx \left(1 - \frac1m\right) e_k, \qquad N'(r) = 1 - \frac1m.
$$

For a double root the error only halves each step: about one binary digit per iteration. The **modified Newton method** $z_{k+1} = z_k - m\,p(z_k)/p'(z_k)$ restores quadratic convergence when $m$ is known.

### 2.4 Real starts stay real

If $p$ has real coefficients and $z_0$ is real, then $p(z_0)$, $p'(z_0)$ and hence $z_1$ are real, and so on forever. Newton's method started on the real line can only find **real** roots. For $x^2 + 1$, a real start wanders chaotically along the real axis and never settles. In $\mathbb{C}$, conjugation symmetry also passes to the Newton map: $N(\bar z) = \overline{N(z)}$, so the basin picture of a real polynomial is symmetric across the real axis.

### 2.5 Basins of attraction and the Newton fractal

The **basin** of a root $r$ is the set of starting points $z_0$ whose iterates converge to $r$. Every basin contains a disk around its root (quadratic convergence), so basins are open sets. Now consider $p(z) = z^3 - 1$, with roots $1$, $\omega = e^{2\pi i/3}$, $\bar\omega = e^{-2\pi i/3}$:

$$
N(z) = z - \frac{z^3 - 1}{3z^2} = \frac{2z^3 + 1}{3z^2}.
$$

- **Symmetry.** $N(\omega z) = \frac{2z^3 + 1}{3\omega^2 z^2} = \omega N(z)$, using $\omega^3 = 1$. Rotating the start by $120°$ rotates the whole orbit by $120°$, so the three basins are congruent copies of each other. With $N(\bar z) = \overline{N(z)}$, the picture has the full symmetry of an equilateral triangle.
- **The trouble spot.** $p'(0) = 0$, so $N(0)$ is undefined ("the tangent is horizontal"). Points near $0$ are thrown far away, and every preimage of $0$ under $N$ is also a point where the iteration breaks down. These preimages accumulate on the basin boundaries.
- **The boundary.** A deep theorem (going back to Julia and Fatou) says that every point on the boundary of one basin is also on the boundary of **all three** basins. You cannot draw a curve separating two basins without the third squeezing in between. This forces an infinitely detailed boundary: the **Newton fractal**, which is the **Julia set** of $N$ (module 07 returns to Julia sets).
- **Sensitivity.** Near a boundary point, starting points arbitrarily close together can converge to different roots. The final answer is a discontinuous function of $z_0$ there.

For other polynomials worse things can happen: the Newton map can have an **attracting cycle** that is not a root, so an open set of starting points never converges to anything. For $p(z) = z^3 - 2z + 2$, $N(0) = 1$ and $N(1) = 0$.

## 3. Worked Examples

### Example 1: two Newton steps for $z^2 + 1$ from $z_0 = 1 + i$

$N(z) = z - \frac{z^2 + 1}{2z} = \frac{z^2 - 1}{2z} = \frac12\left(z - \frac1z\right)$.

- $\frac{1}{1 + i} = \frac{1 - i}{2}$, so $z_1 = \frac12\left((1 + i) - \frac{1 - i}{2}\right) = \frac12\left(\frac12 + \frac32 i\right) = 0.25 + 0.75i$.
- $\frac{1}{0.25 + 0.75i} = \frac{0.25 - 0.75i}{0.0625 + 0.5625} = 0.4 - 1.2i$, so $z_2 = \frac12\big((0.25 + 0.75i) - (0.4 - 1.2i)\big) = -0.075 + 0.975i$.

Errors to the root $i$: $|z_0 - i| = 1$, $|z_1 - i| \approx 0.354$, $|z_2 - i| \approx 0.079$. The next errors are about $3 \times 10^{-3}$, $5 \times 10^{-6}$, $10^{-11}$: digits doubling.

### Example 2: a real start for $z^2 + 1$

With $z_0 = 0.5$: $z_1 = \frac12(0.5 - 2) = -0.75$, $z_2 = \frac12(-0.75 + 1.333) \approx 0.292$, $z_3 \approx -1.569$, $z_4 \approx -0.465$, ... The iterates stay real and bounce around forever: there is no real root to converge to, and Newton cannot leave the real line on its own.

### Example 3: the Newton map of $z^3 - 1$ and its symmetry

$N(z) = \frac{2z^3 + 1}{3z^2}$. Check the rotation symmetry at $z = 2$: $N(2) = \frac{17}{12}$. For $z = 2\omega$: $(2\omega)^3 = 8$, $(2\omega)^2 = 4\omega^2$, so $N(2\omega) = \frac{17}{12\omega^2} = \frac{17}{12}\,\omega$ (because $\frac{1}{\omega^2} = \omega$). So the orbit of $2\omega$ is the orbit of $2$ rotated by $120°$; since $2$ converges to $1$, $2\omega$ converges to $\omega$.

### Example 4: a double root

$p(z) = (z - 1)^2(z + 1)$ has a double root at $1$. From $z_0 = 2 + 0.5i$ the errors $|z_k - 1|$ are approximately $1.12, 0.64, 0.36, 0.19, 0.098, 0.050, 0.025, \dots$: halving each step, as $N'(1) = 1 - \frac12 = \frac12$ predicts. Modified Newton with $m = 2$ converges in a handful of steps (Section 4.3).

### Example 5: a 2-cycle for $z^3 - 2z + 2$

$p'(z) = 3z^2 - 2$. At $z = 0$: $N(0) = 0 - \frac{2}{-2} = 1$. At $z = 1$: $p(1) = 1$, $p'(1) = 1$, $N(1) = 1 - 1 = 0$. The orbit $0 \to 1 \to 0 \to \cdots$ never converges. Moreover $N'(0) = \frac{p(0)p''(0)}{p'(0)^2} = \frac{2 \cdot 0}{4} = 0$, so the cycle is (super)attracting: a whole open region of starting points is pulled into it.

## 4. Python Examples

### 4.1 A Newton iteration that records its path

We iterate until the step is tiny or a maximum number of iterations is reached, and print $|z_k - r|$ to watch the digits double.

In [ ]:
import numpy as np


def newton(f, df, z0, tolerance=1e-14, max_iterations=50):
    path = [complex(z0)]
    z = complex(z0)
    for _ in range(max_iterations):
        step = f(z) / df(z)
        z = z - step
        path.append(z)
        if abs(step) < tolerance:
            break
    return np.array(path)


roots_of_unity = np.exp(2j * np.pi * np.arange(3) / 3)
path = newton(lambda z: z**3 - 1, lambda z: 3 * z**2, z0=-1 + 1j)
for k, z in enumerate(path):
    error = np.min(np.abs(roots_of_unity - z))
    print(f"k = {k}:  z = {z.real:+.15f} {z.imag:+.15f}i   distance to nearest root = {error:.2e}")
print("\nconverged to omega = e^(2 pi i/3):", np.isclose(path[-1], roots_of_unity[1]))

Errors $0.52 \to 0.17 \to 0.026 \to 7 \times 10^{-4} \to 5 \times 10^{-7} \to 3 \times 10^{-13}$: each roughly the square of the previous one (times the constant $|p''/2p'| = 1$ at the roots of $z^3 - 1$).

### 4.2 Real starts stay real

For $z^2 + 1$, a real start never converges; nudging it off the axis by a tiny imaginary part lets it find $\pm i$ (slowly at first, then quickly).

In [ ]:
import numpy as np

f = lambda z: z**2 + 1
df = lambda z: 2 * z

for z0 in [0.5, 0.5 + 1e-6j, 0.5 - 1e-6j]:
    z = complex(z0)
    for _ in range(60):
        z = z - f(z) / df(z)
    print(f"z0 = {complex(z0)}:  after 60 steps z = {z:.6f}   |p(z)| = {abs(f(z)):.1e}")

The real start is still wandering after 60 steps. The perturbed starts end at $+i$ and $-i$ respectively: the upper half-plane is the basin of $i$ and the lower half-plane the basin of $-i$, with the real axis as the boundary.

### 4.3 Multiple roots and the modified method

In [ ]:
import numpy as np

p = lambda z: (z - 1) ** 2 * (z + 1)
dp = lambda z: 2 * (z - 1) * (z + 1) + (z - 1) ** 2

for label, multiplier in [("standard Newton (m = 1)", 1), ("modified Newton (m = 2)", 2)]:
    z = 2 + 0.5j
    errors = []
    for _ in range(8):
        if abs(p(z)) < 1e-300:
            break
        z = z - multiplier * p(z) / dp(z)
        errors.append(abs(z - 1))
    print(label)
    print("   errors:", " ".join(f"{e:.1e}" for e in errors))
    ratios = [errors[k + 1] / errors[k] for k in range(4)]
    print("   first ratios e_(k+1)/e_k:", " ".join(f"{r:.3f}" for r in ratios))

Standard Newton: the ratio settles at $0.5 = 1 - \frac12$. Modified Newton: errors collapse quadratically.

### 4.4 Vectorized Newton over a grid

To draw basins we run Newton on a whole array of starting points at once with numpy, then label each point by the nearest root it reached and record how many steps that took. `np.errstate` silences the division by zero at $z = 0$ (those few points simply become `nan`).

In [ ]:
import numpy as np


def newton_basins(coefficients, x_range, y_range, size, iterations=40, tolerance=1e-6):
    derivative = np.polyder(coefficients)
    roots = np.roots(coefficients)
    roots = roots[np.argsort(np.mod(np.angle(roots), 2 * np.pi))]   # order by angle in [0, 2 pi): 1, omega, conj(omega)
    x = np.linspace(*x_range, size)
    y = np.linspace(*y_range, size)
    Z = x[None, :] + 1j * y[:, None]
    steps_needed = np.full(Z.shape, iterations)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        for k in range(iterations):
            Z = Z - np.polyval(coefficients, Z) / np.polyval(derivative, Z)
            close = np.min(np.abs(Z[..., None] - roots), axis=-1) < tolerance
            steps_needed = np.where(close & (steps_needed == iterations), k + 1, steps_needed)
        distances = np.abs(Z[..., None] - roots)
        label = np.argmin(np.nan_to_num(distances, nan=np.inf), axis=-1)
        converged = np.min(np.nan_to_num(distances, nan=np.inf), axis=-1) < tolerance
    label = np.where(converged, label, -1)
    return roots, label, steps_needed


roots, label, steps = newton_basins([1, 0, 0, -1], (-2, 2), (-2, 2), size=400)
print("roots:", np.round(roots, 4))
for index, root in enumerate(roots):
    print(f"fraction of the square in the basin of {np.round(root, 3)}: {np.mean(label == index):.4f}")
print("fraction not converged after 40 steps:", np.mean(label == -1))
print("median number of steps:", np.median(steps))

By the $120°$ symmetry the three fractions should be equal; they differ slightly only because the square $[-2, 2]^2$ is not itself rotation-symmetric.

## 5. Visualizations

### 5.1 Orbits in the plane

Newton orbits for $z^3 - 1$ from several starting points. Most head straight for the nearest root; the ones starting near the "spokes" (the rays at angles $\pi$, $\pm\frac{\pi}{3}$ between the roots) take detours.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

roots = np.exp(2j * np.pi * np.arange(3) / 3)
colors = ["tab:red", "tab:green", "tab:blue"]
starts = [1.8 + 1.2j, -1 + 1j, -1.5 - 0.3j, 0.3 - 1.5j, -0.6 + 0.05j, 0.25 + 0.45j, -0.2 - 0.9j]

fig, ax = plt.subplots(figsize=(7.5, 7.5))
for z0 in starts:
    z = complex(z0)
    path = [z]
    for _ in range(25):
        z = z - (z**3 - 1) / (3 * z**2)
        path.append(z)
    path = np.array(path)
    destination = np.argmin(np.abs(roots - path[-1]))
    ax.plot(path.real, path.imag, "o-", color=colors[destination], markersize=3, linewidth=1)
    ax.scatter([z0.real], [z0.imag], color=colors[destination], s=60, marker="s", zorder=3)

ax.scatter(roots.real, roots.imag, color="black", s=120, marker="*", zorder=4, label="roots of $z^3 - 1$")
ax.axhline(0, color="black", linewidth=0.6)
ax.axvline(0, color="black", linewidth=0.6)
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-2.2, 2.2)
ax.set_aspect("equal")
ax.set_title("Newton orbits for $z^3 - 1$ (squares = starts, colour = destination)")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower left")
plt.show()

### 5.2 The Newton fractal of $z^3 - 1$

Each pixel is a starting point, coloured by the root it converges to (red: $1$, green: $\omega$, blue: $\bar\omega$) and shaded darker the more iterations it needed. The three big basins surround the roots; along the three spokes they meet in chains of ever-smaller bulbs, and between any two bulbs of different colours sits a bulb of the third colour.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def newton_basins(coefficients, x_range, y_range, size, iterations=40, tolerance=1e-6):
    derivative = np.polyder(coefficients)
    roots = np.roots(coefficients)
    roots = roots[np.argsort(np.mod(np.angle(roots), 2 * np.pi))]   # order by angle in [0, 2 pi): 1, omega, conj(omega)
    x = np.linspace(*x_range, size)
    y = np.linspace(*y_range, size)
    Z = x[None, :] + 1j * y[:, None]
    steps_needed = np.full(Z.shape, iterations)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        for k in range(iterations):
            Z = Z - np.polyval(coefficients, Z) / np.polyval(derivative, Z)
            close = np.min(np.abs(Z[..., None] - roots), axis=-1) < tolerance
            steps_needed = np.where(close & (steps_needed == iterations), k + 1, steps_needed)
        distances = np.nan_to_num(np.abs(Z[..., None] - roots), nan=np.inf)
    label = np.where(np.min(distances, axis=-1) < tolerance, np.argmin(distances, axis=-1), -1)
    return roots, label, steps_needed


def basin_image(label, steps, palette):
    shade = 1.0 - 0.75 * np.clip(steps / 25, 0, 1)        # darker = slower
    image = np.zeros(label.shape + (3,))
    for index, color in enumerate(palette):
        mask = label == index
        image[mask] = np.array(color) * shade[mask, None]
    return image                                         # non-converged points stay black


palette = [(0.89, 0.29, 0.20), (0.30, 0.69, 0.29), (0.22, 0.49, 0.72)]
extent = (-2, 2, -2, 2)
roots, label, steps = newton_basins([1, 0, 0, -1], extent[:2], extent[2:], size=800)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(basin_image(label, steps, palette), extent=extent, origin="lower")
ax.scatter(roots.real, roots.imag, color="white", edgecolor="black", s=80, zorder=3)
ax.set_aspect("equal")
ax.set_title("Newton fractal of $z^3 - 1$ (800 x 800 starting points)")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
plt.show()

### 5.3 Zooming into a boundary

Zoom into the box $[-0.9, -0.3] \times [-0.3, 0.3]$ on the negative real axis, and then further into a small box on its upper boundary. The same pattern of three interleaved colours reappears at every scale: this self-similarity is what makes the boundary a fractal.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def newton_basins(coefficients, x_range, y_range, size, iterations=60, tolerance=1e-6):
    derivative = np.polyder(coefficients)
    roots = np.roots(coefficients)
    roots = roots[np.argsort(np.mod(np.angle(roots), 2 * np.pi))]   # order by angle in [0, 2 pi): 1, omega, conj(omega)
    x = np.linspace(*x_range, size)
    y = np.linspace(*y_range, size)
    Z = x[None, :] + 1j * y[:, None]
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        for _ in range(iterations):
            Z = Z - np.polyval(coefficients, Z) / np.polyval(derivative, Z)
        distances = np.nan_to_num(np.abs(Z[..., None] - roots), nan=np.inf)
    return np.where(np.min(distances, axis=-1) < tolerance, np.argmin(distances, axis=-1), -1)


palette = np.array([(0.89, 0.29, 0.20), (0.30, 0.69, 0.29), (0.22, 0.49, 0.72), (0, 0, 0)])
windows = [(-0.9, -0.3, -0.3, 0.3), (-0.62, -0.5, 0.12, 0.24)]

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, (x0, x1, y0, y1) in zip(axes, windows):
    label = newton_basins([1, 0, 0, -1], (x0, x1), (y0, y1), size=600)
    ax.imshow(palette[label], extent=(x0, x1, y0, y1), origin="lower")   # label -1 picks the last row: black
    ax.set_aspect("equal")
    ax.set_title(f"Re in [{x0}, {x1}], Im in [{y0}, {y1}]")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
axes[0].add_patch(plt.Rectangle((windows[1][0], windows[1][2]), windows[1][1] - windows[1][0],
                                windows[1][3] - windows[1][2], fill=False, edgecolor="white", linewidth=1.5))
plt.suptitle("Zooming into the basin boundary of $z^3 - 1$: the white box on the left is magnified on the right")
plt.tight_layout()
plt.show()

### 5.4 Sensitivity on the boundary

Take a short horizontal segment of starting points crossing the boundary and record the destination of each. Inside a basin the destination is constant; on the boundary it flips between all three roots in ever finer bands. Then bisect to locate a boundary point and show that two starting points $10^{-12}$ apart can end at different roots.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

roots = np.exp(2j * np.pi * np.arange(3) / 3)


def destination(z0, iterations=80):
    z = np.asarray(z0, dtype=complex)
    with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
        for _ in range(iterations):
            z = z - (z**3 - 1) / (3 * z**2)
        distances = np.nan_to_num(np.abs(z[..., None] - roots), nan=np.inf)
    return np.where(np.min(distances, axis=-1) < 1e-8, np.argmin(distances, axis=-1), -1)


# A horizontal segment at Im = 0.15 crossing the spoke on the negative real axis
x = np.linspace(-1.2, 0.2, 4000)
segment = x + 0.15j
labels = destination(segment)

palette = np.array([(0.89, 0.29, 0.20), (0.30, 0.69, 0.29), (0.22, 0.49, 0.72), (0, 0, 0)])
fig, ax = plt.subplots(figsize=(11, 2.2))
ax.imshow(palette[labels][None, :, :], extent=(x[0], x[-1], 0, 1), aspect="auto")
ax.set_yticks([])
ax.set_xlabel("Re $z_0$ (with Im $z_0$ = 0.15)")
ax.set_title("Destination of Newton's method along a segment: red = 1, green = $\\omega$, blue = $\\bar\\omega$")
plt.show()

# Bisection between two neighbouring starting points with different destinations
change = np.nonzero(labels[1:] != labels[:-1])[0][0]
left, right = segment[change], segment[change + 1]
label_left = destination(left)
for _ in range(60):
    middle = (left + right) / 2
    if destination(middle) == label_left:
        left = middle
    else:
        right = middle
    if abs(right - left) < 1e-12:
        break
print(f"two starts {abs(right - left):.1e} apart near z0 = {left:.12f}")
print("   destination of the left one :", np.round(roots[destination(left)], 4))
print("   destination of the right one:", np.round(roots[destination(right)], 4))

### 5.5 When Newton fails: $z^3 - 2z + 2$

For this polynomial the 2-cycle $0 \leftrightarrow 1$ is attracting. Starting points in the black regions are captured by the cycle and never converge to any root. These black regions are open sets; this is not a rounding artefact.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

coefficients = [1, 0, -2, 2]
derivative = np.polyder(coefficients)
roots = np.roots(coefficients)

size = 600
x = np.linspace(-2.5, 2.5, size)
Z = x[None, :] + 1j * x[:, None]
with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    for _ in range(60):
        Z = Z - np.polyval(coefficients, Z) / np.polyval(derivative, Z)
    distances = np.nan_to_num(np.abs(Z[..., None] - roots), nan=np.inf)
label = np.where(np.min(distances, axis=-1) < 1e-6, np.argmin(distances, axis=-1), -1)

palette = np.array([(0.89, 0.29, 0.20), (0.30, 0.69, 0.29), (0.22, 0.49, 0.72), (0, 0, 0)])
fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(palette[label], extent=(-2.5, 2.5, -2.5, 2.5), origin="lower")
ax.scatter(roots.real, roots.imag, color="white", edgecolor="black", s=80, zorder=3, label="roots")
ax.scatter([0, 1], [0, 0], color="yellow", edgecolor="black", s=60, marker="D", zorder=3, label="2-cycle {0, 1}")
ax.set_aspect("equal")
ax.set_title("Newton basins of $z^3 - 2z + 2$ (black: trapped by the cycle)")
ax.set_xlabel("Re")
ax.set_ylabel("Im")
ax.legend(loc="lower left")
plt.show()
print("fraction of the square that never converges:", round(float(np.mean(label == -1)), 4))

## 6. Exercises

### Basic

1. Do one Newton step for $p(z) = z^2 + 4$ from $z_0 = 1 + i$ by hand. Which root is $z_1$ closer to?
2. Write the Newton map $N(z)$ for $p(z) = z^3 - 8$ as a single fraction. What are its fixed points?
3. Explain in two sentences why Newton's method started at a real number can never converge to $1 + i$ when applied to $z^2 - 2z + 2$.

### Intermediate

4. **Draw it first, then compute.** For $z^3 - 1$, use the symmetries $N(\omega z) = \omega N(z)$ and $N(\bar z) = \overline{N(z)}$ to predict the destinations of $z_0 = 2$, $2\omega$, $2\bar\omega$ and $-2$, and mark them on a sketch of the three roots. Then verify numerically.
5. Show algebraically that $N(\bar z) = \overline{N(z)}$ for any real polynomial, and explain what this means for the Newton fractal of a real polynomial.
6. For $z^2 + 1$ from $z_0 = 1 + i$, count the iterations needed for $|z_k - i| < 10^{-12}$ and check that $|e_{k+1}| / |e_k|^2$ approaches $\left|\frac{p''(i)}{2p'(i)}\right| = \frac12$.

### Advanced

7. Let $p(z) = (z^2 + 1)^2$. Show $N'(\pm i) = \frac12$, observe linear convergence numerically from $z_0 = 0.5 + 1.5i$, and fix it with the modified method.
8. For $p(z) = z^3 - 2z + 2$, verify the 2-cycle $0 \leftrightarrow 1$ and show it is attracting by computing $(N \circ N)'(0) = N'(1)N'(0)$.
9. Newton's method for $z^2 - a$ gives $N(z) = \frac12\left(z + \frac{a}{z}\right)$, Heron's ancient method for square roots. Use it to compute $\sqrt{3 + 4i}$ from $z_0 = 1$, and determine (by the argument of 4.2) which half-plane of starting points leads to which square root.

## 7. Solutions

1. $N(z) = z - \frac{z^2 + 4}{2z} = \frac12\left(z - \frac4z\right)$. $\frac{4}{1 + i} = 2(1 - i) = 2 - 2i$, so $z_1 = \frac12\big((1 + i) - (2 - 2i)\big) = -0.5 + 1.5i$. Distances: $|z_1 - 2i| = |{-0.5} - 0.5i| \approx 0.71$, $|z_1 + 2i| \approx 3.54$; closer to $2i$.
2. $N(z) = z - \frac{z^3 - 8}{3z^2} = \frac{2z^3 + 8}{3z^2}$. Fixed points: $N(z) = z \iff 2z^3 + 8 = 3z^3 \iff z^3 = 8$, the three cube roots $2, 2\omega, 2\bar\omega$.
3. The coefficients are real, so a real $z_k$ gives real $p(z_k)$ and $p'(z_k)$ and hence a real $z_{k+1}$. The iterates never leave the real line, which is at distance $1$ from $1 + i$.
4. On the real axis iterates stay real, and the only real root is $1$, so $2 \to 1$ and $-2 \to 1$ (it first jumps to $N(-2) = \frac{-15}{12} = -1.25$, then $\approx -0.62$, then far to the right, then home to $1$). By rotation, $2\omega \to \omega$ and $2\bar\omega \to \bar\omega$. Sketch: three arrows from the points at radius $2$ to the roots on the unit circle, plus $-2$ travelling across to $1$.
5. If $p$ is real then so is $p'$, so $p(\bar z) = \overline{p(z)}$ and $p'(\bar z) = \overline{p'(z)}$. Hence $N(\bar z) = \bar z - \frac{\overline{p(z)}}{\overline{p'(z)}} = \overline{z - \frac{p(z)}{p'(z)}} = \overline{N(z)}$. The orbit of $\bar z_0$ is the mirror image of the orbit of $z_0$, so the basin of $\bar r$ is the mirror image of the basin of $r$, and the fractal is symmetric across the real axis.
6. Seven iterations: $|e_6| \approx 1.3 \times 10^{-11}$ is not yet below $10^{-12}$, but $|e_7| \approx 10^{-22}$ is. The ratios $|e_{k+1}|/|e_k|^2$ for $k = 1, \dots, 4$ are about $0.63, 0.51, 0.50, 0.50$ (later ones are spoiled by rounding): they tend to $\frac12$ because $p''(i)/(2p'(i)) = 2/(4i)$ has modulus $\frac12$.
7. $\pm i$ are double roots, so $N'(\pm i) = 1 - \frac12 = \frac12$. Numerically the errors halve each step. With $z_{k+1} = z_k - 2\frac{p(z_k)}{p'(z_k)} = z_k - \frac{z_k^2 + 1}{2z_k}$, which is just Newton for $z^2 + 1$, convergence is quadratic again.
8. $N(0) = 1$, $N(1) = 0$ (Worked Example 5). $N'(z) = \frac{p(z)p''(z)}{p'(z)^2}$ with $p''(z) = 6z$: $N'(0) = \frac{2 \cdot 0}{4} = 0$ and $N'(1) = \frac{1 \cdot 6}{1} = 6$. By the chain rule $(N \circ N)'(0) = N'(N(0))\,N'(0) = 6 \cdot 0 = 0$, so $0$ is a superattracting fixed point of $N \circ N$; nearby starts are pulled into the cycle.
9. From $z_0 = 1$: $z_1 = 2 + 2i$, $z_2 = 1.875 + 1.125i$, $z_3 \approx 1.996 + 0.994i$, then $2 + i$ to machine precision: $\sqrt{3 + 4i} = 2 + i$ (principal root). The substitution $z = (2 + i)u$ turns Newton for $z^2 - (3 + 4i)$ into Newton for $u^2 - 1$, whose basins are the half-planes $\operatorname{Re} u > 0$ (to $1$) and $\operatorname{Re} u < 0$ (to $-1$). Mapping back, the basins of $\pm(2 + i)$ are the two half-planes separated by the line through $0$ perpendicular to $2 + i$: starts with $\operatorname{Re}\big(z_0\,\overline{(2 + i)}\big) > 0$ go to $2 + i$, the others to $-2 - i$.

Numerical checks:

In [ ]:
import numpy as np


def iterate(f, df, z0, steps):
    path = [complex(z0)]
    for _ in range(steps):
        z = path[-1]
        path.append(z - f(z) / df(z))
    return np.array(path)


print("Ex 1:", iterate(lambda z: z**2 + 4, lambda z: 2 * z, 1 + 1j, 1)[-1])

roots = np.exp(2j * np.pi * np.arange(3) / 3)
for z0 in [2, 2 * roots[1], 2 * roots[2], -2]:
    end = iterate(lambda z: z**3 - 1, lambda z: 3 * z**2, z0, 40)[-1]
    print(f"Ex 4: z0 = {np.round(complex(z0), 3)} -> {np.round(end, 6)}")

errors = np.abs(iterate(lambda z: z**2 + 1, lambda z: 2 * z, 1 + 1j, 7) - 1j)
print("Ex 6: errors", " ".join(f"{e:.1e}" for e in errors))
print("      ratios |e_k+1|/|e_k|^2", " ".join(f"{errors[k + 1] / errors[k] ** 2:.3f}" for k in range(1, 5)))

p = lambda z: (z**2 + 1) ** 2
dp = lambda z: 4 * z * (z**2 + 1)
standard = np.abs(iterate(p, dp, 0.5 + 1.5j, 8) - 1j)
modified = np.abs(iterate(p, lambda z: dp(z) / 2, 0.5 + 1.5j, 5) - 1j)
print("Ex 7: standard ratios", " ".join(f"{standard[k + 1] / standard[k]:.3f}" for k in range(6)))
print("      modified errors", " ".join(f"{e:.1e}" for e in modified))

N = lambda z: z - (z**3 - 2 * z + 2) / (3 * z**2 - 2)
dN = lambda z: (z**3 - 2 * z + 2) * 6 * z / (3 * z**2 - 2) ** 2
print("Ex 8: N(0) =", N(0), " N(1) =", N(1), " N'(0) * N'(1) =", dN(0) * dN(1))

print("Ex 9:", np.round(iterate(lambda z: z**2 - (3 + 4j), lambda z: 2 * z, 1, 6), 6))
print("      start -1 + 0.5i (Re(z0 * conj(2+i)) < 0) ->",
      np.round(iterate(lambda z: z**2 - (3 + 4j), lambda z: 2 * z, -1 + 0.5j, 40)[-1], 6))

## 8. Conclusion

- Newton's method in $\mathbb{C}$: $z_{k+1} = z_k - p(z_k)/p'(z_k)$, the zero of the local linear approximation, exactly as on the real line.
- Near a simple root the error is squared each step ($N'(r) = 0$); near a root of multiplicity $m$ it shrinks only by the factor $1 - \frac1m$, and the modified step $-m\,p/p'$ restores quadratic convergence.
- Real polynomials with real starting points stay on the real line; complex roots need complex starts.
- Basins of attraction are open sets whose common boundary is a fractal (the Julia set of $N$). Each boundary point touches all three basins of $z^3 - 1$, and starts $10^{-12}$ apart can converge to different roots.
- Newton can fail outright: $z^3 - 2z + 2$ has an attracting 2-cycle that captures an open set of starting points.
- Symmetries of $p$ (rotation, conjugation) become symmetries of the Newton map and of its fractal.

**Next:** notebook 04, [Partial fractions over C](04_partial_fractions_over_c.ipynb), uses the complete factorization into linear factors to split rational functions into simple pieces $\frac{A}{z - z_k}$, the algebraic preparation for residues.